In [2]:
import os 

# set directory to parent directory
os.chdir("..")

# print current working directory
print("Current Working Directory:", os.getcwd())


Current Working Directory: /Users/jorgegoncalves/Library/CloudStorage/OneDrive-Persönlich/Dokumente/Universität/Master/HS23/Master_Thesis/Code/treevae


In [3]:

from FID.fid_score import calculate_fid, get_precomputed_fid_scores_path, save_fid_stats_as_dict
import gzip
import numpy as np
import os
from PIL import Image
import pickle
import torch
import yaml
import argparse
from pathlib import Path
from utils.data_utils import get_data, get_gen
from utils.utils import reset_random_seeds, prepare_config
from models.losses import loss_reconstruction_binary, loss_reconstruction_mse

In [16]:
path = '/Users/jorgegoncalves/Library/CloudStorage/OneDrive-Persönlich/Dokumente/Universität/Master/HS23/Master_Thesis/Code/treevae/models/experiments/'

dataset = 'mnist/'
ex_name = '20240221-150914_9e40a'

dataset = 'cifar10/'
ex_name = '20240303-144417_f2db4'

checkpoint_path = path+dataset+ex_name
with open(checkpoint_path + "/config.yaml", 'r') as stream:
    configs = yaml.load(stream,Loader=yaml.Loader)
print(configs)

from utils.data_utils import get_data, get_gen

# load data
trainset, trainset_eval, testset = get_data(configs)
gen_train = get_gen(trainset, configs, validation=False, shuffle=False)
gen_train_eval = get_gen(trainset_eval, configs, validation=True, shuffle=False)
gen_test = get_gen(testset, configs, validation=True, shuffle=False)
gen_train_eval_iter = iter(gen_train_eval)
gen_test_iter = iter(gen_test)
y_train = trainset_eval.dataset.targets.clone().detach()[trainset_eval.indices].numpy()
y_test = testset.dataset.targets.clone().detach()[testset.indices].numpy()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "mps"

{'data': {'data_name': 'cifar10', 'num_clusters_data': 10}, 'globals': {'config_name': 'cifar10', 'eager_mode': True, 'results_dir': PosixPath('/cluster/home/jogoncalves/treevae/models/experiments'), 'save_model': True, 'seed': 42, 'wandb_logging': 'offline'}, 'parser': {}, 'run_name': 'cifar10_mlp', 'training': {'activation': 'mse', 'aug_decisions_weight': 100, 'augment': True, 'augmentation_method': ['InfoNCE', 'instancewise_full'], 'batch_size': 256, 'compute_fid': True, 'compute_ll': False, 'decay_kl': 0.01, 'decay_lr': 0.1, 'decay_stepsize': 100, 'encoder': 'cnn2', 'grow': True, 'initial_depth': 1, 'inp_shape': 3072, 'kl_start': 0.01, 'latent_dim': [64, 64, 64, 64, 64, 64, 64], 'lr': 0.001, 'mlp_layers': [4096, 512, 512, 512, 512, 512, 512], 'num_clusters_tree': 10, 'num_epochs': 1, 'num_epochs_finetuning': 0, 'num_epochs_intermediate_fulltrain': 0, 'num_epochs_smalltree': 1, 'prune': True, 'save_images': True, 'weight_decay': 1e-05}}
Files already downloaded and verified
Files al

In [17]:
ddpm_samples_path = '../from_cluster/results/' + dataset + 'seed_1/ddpm/sample/'
ddpm_reconstructions_path = '../from_cluster/results/' + dataset + 'seed_1/ddpm/recons/'

vae_samples_path = '../from_cluster/results/' + dataset + 'seed_1/vae/sample/'
vae_reconstructions_path = '../from_cluster/results/' + dataset + 'seed_1/vae/recons/'

In [18]:
# load all images from the path and create a numpy array
ddpm_samples = []
for filename in os.listdir(ddpm_samples_path):
    if filename.endswith(".png"):
        img = Image.open(ddpm_samples_path + filename)
        img = np.array(img)
        ddpm_samples.append(img)
ddpm_samples = np.array(ddpm_samples)
print("Nb. of ddpm samples: ", len(ddpm_samples), "\n")

ddpm_reconstructions = []
for filename in os.listdir(ddpm_reconstructions_path):
    if filename.endswith(".png"):
        img = Image.open(ddpm_reconstructions_path + filename)
        img = np.array(img)
        ddpm_reconstructions.append(img)
ddpm_reconstructions = np.array(ddpm_reconstructions)
print("Nb. of ddpm reconstructions: ", len(ddpm_reconstructions), "\n")

vae_samples = []
for filename in os.listdir(vae_samples_path):
    if filename.endswith(".png"):
        img = Image.open(vae_samples_path + filename)
        img = np.array(img)
        vae_samples.append(img)
vae_samples = np.array(vae_samples)
print("Nb. of vae samples: ", len(vae_samples), "\n")

vae_reconstructions = []
for filename in os.listdir(vae_reconstructions_path):
    if filename.endswith(".png"):
        img = Image.open(vae_reconstructions_path + filename)
        img = np.array(img)
        vae_reconstructions.append(img)
vae_reconstructions = np.array(vae_reconstructions)
print("Nb. of vae reconstructions: ", len(vae_reconstructions), "\n")



Nb. of ddpm samples:  10000 

Nb. of ddpm reconstructions:  10000 

Nb. of vae samples:  10000 

Nb. of vae reconstructions:  10000 



In [19]:
# precompute or load fid scores for train and test
data_stats_train = get_precomputed_fid_scores_path(trainset.dataset.data, configs['data']['data_name'],
                                                subset="train", device=device)
data_stats_test = get_precomputed_fid_scores_path(testset.dataset.data, configs['data']['data_name'], subset="test",
                                                device=device)

---

VAE SAMPLES

In [20]:
# precompute FID scores for generated images
stats_generations = save_fid_stats_as_dict(vae_samples, batch_size=256, device=device, dims=2048)
train_FID_generations = calculate_fid([data_stats_train, stats_generations], batch_size=256, device=device, dims=2048)
test_FID_generations = calculate_fid([data_stats_test, stats_generations], batch_size=256, device=device, dims=2048)
print("FID score for generated images compared to train set:", train_FID_generations)
print("FID score for generated images compared to test set:", test_FID_generations)

Saving FID statistics


100%|██████████| 40/40 [00:52<00:00,  1.30s/it]


FID score for generated images compared to train set: 207.80528852902063
FID score for generated images compared to test set: 208.84495887358773


DDPM SAMPLES

In [21]:
# precompute FID scores for generated images
stats_generations = save_fid_stats_as_dict(ddpm_samples, batch_size=256, device=device, dims=2048)
train_FID_generations = calculate_fid([data_stats_train, stats_generations], batch_size=256, device=device, dims=2048)
test_FID_generations = calculate_fid([data_stats_test, stats_generations], batch_size=256, device=device, dims=2048)
print("FID score for generated images compared to train set:", train_FID_generations)
print("FID score for generated images compared to test set:", test_FID_generations)


Saving FID statistics


100%|██████████| 40/40 [00:52<00:00,  1.31s/it]


FID score for generated images compared to train set: 20.767625425869255
FID score for generated images compared to test set: 22.80768232044005


---

VAE RECONSTRUCTIONS

In [22]:
# precompute FID scores for generated images
stats_generations = save_fid_stats_as_dict(vae_reconstructions, batch_size=256, device=device, dims=2048)
train_FID_generations = calculate_fid([data_stats_train, stats_generations], batch_size=256, device=device, dims=2048)
test_FID_generations = calculate_fid([data_stats_test, stats_generations], batch_size=256, device=device, dims=2048)
print("FID score for generated images compared to train set:", train_FID_generations)
print("FID score for generated images compared to test set:", test_FID_generations)

Saving FID statistics


100%|██████████| 40/40 [00:51<00:00,  1.30s/it]


FID score for generated images compared to train set: 193.19771977090804
FID score for generated images compared to test set: 194.26382854522302


DDPM RECONSTRUCTIONS

In [15]:
# precompute FID scores for generated images
stats_generations = save_fid_stats_as_dict(ddpm_reconstructions, batch_size=256, device=device, dims=2048)
train_FID_generations = calculate_fid([data_stats_train, stats_generations], batch_size=256, device=device, dims=2048)
test_FID_generations = calculate_fid([data_stats_test, stats_generations], batch_size=256, device=device, dims=2048)
print("FID score for generated images compared to train set:", train_FID_generations)
print("FID score for generated images compared to test set:", test_FID_generations)

Saving FID statistics


100%|██████████| 40/40 [00:52<00:00,  1.31s/it]


FID score for generated images compared to train set: 1.0107779633030702
FID score for generated images compared to test set: 1.459633255183519


---
---

# MNIST

In [28]:

dataset = 'mnist/'

# use config of some trained mnist model to load data correctly
ex_name = '20240221-150914_9e40a'

checkpoint_path = path+dataset+ex_name
with open(checkpoint_path + "/config.yaml", 'r') as stream:
    configs = yaml.load(stream,Loader=yaml.Loader)

# load data
trainset, trainset_eval, testset = get_data(configs)
gen_train = get_gen(trainset, configs, validation=False, shuffle=False)
gen_train_eval = get_gen(trainset_eval, configs, validation=True, shuffle=False)
gen_test = get_gen(testset, configs, validation=True, shuffle=False)
gen_train_eval_iter = iter(gen_train_eval)
gen_test_iter = iter(gen_test)
y_train = trainset_eval.dataset.targets.clone().detach()[trainset_eval.indices].numpy()
y_test = testset.dataset.targets.clone().detach()[testset.indices].numpy()

# setup device, mps is for M1 or M2 macs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "mps"

# precompute or load fid scores for train and test
data_stats_train = get_precomputed_fid_scores_path(trainset.dataset.data, configs['data']['data_name'],
                                                subset="train", device=device)
data_stats_test = get_precomputed_fid_scores_path(testset.dataset.data, configs['data']['data_name'], subset="test",
                                                device=device)

# paths to the generated images
base_path = '../from_cluster/results/'
seed_list = ["seed_1", "seed_2", "seed_3", "seed_4", "seed_5", "seed_6", "seed_7", "seed_8", "seed_9", "seed_10"]

# lists to store FID scores for each seed
mnist_train_FID_generations_ddpm = []
mnist_test_FID_generations_ddpm = []
mnist_train_FID_reconstructions_ddpm = []
mnist_test_FID_reconstructions_ddpm = []

for i, seed in enumerate(seed_list):
    ddpm_samples_path = base_path + dataset + seed + '/ddpm/sample/'
    ddpm_reconstructions_path = base_path + dataset + seed + '/ddpm/recons/'

    # load all images from the path and create a numpy array
    ddpm_samples = []
    for filename in os.listdir(ddpm_samples_path):
        if filename.endswith(".png"):
            img = Image.open(ddpm_samples_path + filename)
            img = np.array(img)
            ddpm_samples.append(img)
    ddpm_samples = np.array(ddpm_samples)

    ddpm_reconstructions = []
    for filename in os.listdir(ddpm_reconstructions_path):
        if filename.endswith(".png"):
            img = Image.open(ddpm_reconstructions_path + filename)
            img = np.array(img)
            ddpm_reconstructions.append(img)
    ddpm_reconstructions = np.array(ddpm_reconstructions)

    # precompute FID scores for generated and reconstructed images of DDPM

    # DDPM samples
    ddpm_stats_generations = save_fid_stats_as_dict(ddpm_samples, batch_size=256, device=device, dims=2048)
    train_FID_gen_ddpm = calculate_fid([data_stats_train, ddpm_stats_generations], batch_size=256, device=device, dims=2048)
    test_FID_gen_ddpm = calculate_fid([data_stats_test, ddpm_stats_generations], batch_size=256, device=device, dims=2048)
    mnist_train_FID_generations_ddpm.append(train_FID_gen_ddpm)
    mnist_test_FID_generations_ddpm.append(test_FID_gen_ddpm)

    # DDPM reconstructions
    ddpm_stats_reconstructions = save_fid_stats_as_dict(ddpm_reconstructions, batch_size=256, device=device, dims=2048)
    train_FID_recon_ddpm = calculate_fid([data_stats_train, ddpm_stats_reconstructions], batch_size=256, device=device, dims=2048)
    test_FID_recon_ddpm = calculate_fid([data_stats_test, ddpm_stats_reconstructions], batch_size=256, device=device, dims=2048)
    mnist_train_FID_reconstructions_ddpm.append(train_FID_recon_ddpm)
    mnist_test_FID_reconstructions_ddpm.append(test_FID_recon_ddpm)



Saving FID statistics


100%|██████████| 40/40 [01:00<00:00,  1.51s/it]


Saving FID statistics


100%|██████████| 40/40 [00:51<00:00,  1.29s/it]


Saving FID statistics


...

100%|██████████| 40/40 [00:51<00:00,  1.28s/it]


Saving FID statistics


100%|██████████| 40/40 [00:51<00:00,  1.30s/it]


Saving FID statistics


100%|██████████| 40/40 [00:50<00:00,  1.27s/it]


In [29]:
print("\nFID scores for DDPM samples compared to train set:\n", mnist_train_FID_generations_ddpm)
print("\nFID scores for DDPM samples compared to test set:\n", mnist_test_FID_generations_ddpm)
print("\nFID scores for DDPM reconstructions compared to train set:\n", mnist_train_FID_reconstructions_ddpm)
print("\nFID scores for DDPM reconstructions compared to test set:\n", mnist_test_FID_reconstructions_ddpm)


FID scores for DDPM samples compared to train set:
 [15.140318726993826, 9.255691065784447, 9.761814690617058, 25.50937665180774, 14.087065564961904, 13.095041201365234, 19.22885894095151, 6.383049283513856, 19.75140006010173, 11.981913493474678]

FID scores for DDPM samples compared to test set:
 [17.05497295986737, 10.848314210920876, 11.356067525243446, 27.791030113934283, 15.814767393982919, 14.9979892232071, 21.26501304919185, 7.9642565305629205, 21.810952872992345, 13.523891242818479]

FID scores for DDPM reconstructions compared to train set:
 [1.0107779633030702, 0.8964708068821778, 0.9695426766013497, 0.9599052292478234, 0.9742328694040339, 0.9697083703799763, 1.0231575495412812, 1.0231670567287665, 1.1033278662509929, 0.9016469721154294]

FID scores for DDPM reconstructions compared to test set:
 [1.459633255183519, 1.481036880160076, 1.4744896525787397, 1.4295356992947745, 1.4286686128590418, 1.541056758383263, 1.5084492303184618, 1.4456969142233618, 1.5622130793569227, 1.5

In [30]:
# compute average and standard deviation of FID scores for DDPM 
print("MNIST")
print("-----------------------------------")
print("Reconstructions")
print("\nAverage FID score for DDPM reconstructions compared to test set:", np.mean(mnist_test_FID_reconstructions_ddpm))
print("\nStandard deviation FID score for DDPM reconstructions compared to test set:", np.std(mnist_test_FID_reconstructions_ddpm))
print("-----------------------------------")
print("Generations")
print("\nAverage FID score for DDPM samples compared to test set:", np.mean(mnist_test_FID_generations_ddpm))
print("\nStandard deviation FID score for DDPM samples compared to test set:", np.std(mnist_test_FID_generations_ddpm))

MNIST
-----------------------------------
Reconstructions

Average FID score for DDPM reconstructions compared to test set: 1.4850161198209775

Standard deviation FID score for DDPM reconstructions compared to test set: 0.044045472595191655
-----------------------------------
Generations

Average FID score for DDPM samples compared to test set: 16.24272551227216

Standard deviation FID score for DDPM samples compared to test set: 5.664700678320154


# FashionMNIST

In [32]:

dataset = 'fmnist/'

# use config of some trained fmnist model to load data correctly
ex_name = '20240303-144050_23df4'

checkpoint_path = path+dataset+ex_name
with open(checkpoint_path + "/config.yaml", 'r') as stream:
    configs = yaml.load(stream,Loader=yaml.Loader)

# load data
trainset, trainset_eval, testset = get_data(configs)
gen_train = get_gen(trainset, configs, validation=False, shuffle=False)
gen_train_eval = get_gen(trainset_eval, configs, validation=True, shuffle=False)
gen_test = get_gen(testset, configs, validation=True, shuffle=False)
gen_train_eval_iter = iter(gen_train_eval)
gen_test_iter = iter(gen_test)
y_train = trainset_eval.dataset.targets.clone().detach()[trainset_eval.indices].numpy()
y_test = testset.dataset.targets.clone().detach()[testset.indices].numpy()

# setup device, mps is for M1 or M2 macs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "mps"

# precompute or load fid scores for train and test
data_stats_train = get_precomputed_fid_scores_path(trainset.dataset.data, configs['data']['data_name'],
                                                subset="train", device=device)
data_stats_test = get_precomputed_fid_scores_path(testset.dataset.data, configs['data']['data_name'], subset="test",
                                                device=device)

# paths to the generated images
base_path = '../from_cluster/results/'
seed_list = ["seed_1", "seed_2", "seed_3", "seed_4", "seed_5", "seed_6", "seed_7", "seed_8", "seed_9", "seed_10"]

# lists to store FID scores for each seed
fmnist_train_FID_generations_ddpm = []
fmnist_test_FID_generations_ddpm = []
fmnist_train_FID_reconstructions_ddpm = []
fmnist_test_FID_reconstructions_ddpm = []

for i, seed in enumerate(seed_list):
    ddpm_samples_path = base_path + dataset + seed + '/ddpm/sample/'
    ddpm_reconstructions_path = base_path + dataset + seed + '/ddpm/recons/'

    # load all images from the path and create a numpy array
    ddpm_samples = []
    for filename in os.listdir(ddpm_samples_path):
        if filename.endswith(".png"):
            img = Image.open(ddpm_samples_path + filename)
            img = np.array(img)
            ddpm_samples.append(img)
    ddpm_samples = np.array(ddpm_samples)

    ddpm_reconstructions = []
    for filename in os.listdir(ddpm_reconstructions_path):
        if filename.endswith(".png"):
            img = Image.open(ddpm_reconstructions_path + filename)
            img = np.array(img)
            ddpm_reconstructions.append(img)
    ddpm_reconstructions = np.array(ddpm_reconstructions)

    # precompute FID scores for generated and reconstructed images of VAE and DDPM

    # DDPM samples
    ddpm_stats_generations = save_fid_stats_as_dict(ddpm_samples, batch_size=256, device=device, dims=2048)
    train_FID_gen_ddpm = calculate_fid([data_stats_train, ddpm_stats_generations], batch_size=256, device=device, dims=2048)
    test_FID_gen_ddpm = calculate_fid([data_stats_test, ddpm_stats_generations], batch_size=256, device=device, dims=2048)
    fmnist_train_FID_generations_ddpm.append(train_FID_gen_ddpm)
    fmnist_test_FID_generations_ddpm.append(test_FID_gen_ddpm)

    # DDPM reconstructions
    ddpm_stats_reconstructions = save_fid_stats_as_dict(ddpm_reconstructions, batch_size=256, device=device, dims=2048)
    train_FID_recon_ddpm = calculate_fid([data_stats_train, ddpm_stats_reconstructions], batch_size=256, device=device, dims=2048)
    test_FID_recon_ddpm = calculate_fid([data_stats_test, ddpm_stats_reconstructions], batch_size=256, device=device, dims=2048)
    fmnist_train_FID_reconstructions_ddpm.append(train_FID_recon_ddpm)
    fmnist_test_FID_reconstructions_ddpm.append(test_FID_recon_ddpm)

Saving FID statistics


100%|██████████| 40/40 [00:51<00:00,  1.30s/it]


Saving FID statistics


100%|██████████| 40/40 [00:51<00:00,  1.29s/it]


Saving FID statistics


...

100%|██████████| 40/40 [00:51<00:00,  1.29s/it]


Saving FID statistics


100%|██████████| 40/40 [00:50<00:00,  1.27s/it]


Saving FID statistics


100%|██████████| 40/40 [00:51<00:00,  1.29s/it]


In [33]:
print("\nFID scores for DDPM samples compared to train set:\n", fmnist_train_FID_generations_ddpm)
print("\nFID scores for DDPM samples compared to test set:\n", fmnist_test_FID_generations_ddpm)
print("\nFID scores for DDPM reconstructions compared to train set:\n", fmnist_train_FID_reconstructions_ddpm)
print("\nFID scores for DDPM reconstructions compared to test set:\n", fmnist_test_FID_reconstructions_ddpm)


FID scores for DDPM samples compared to train set:
 [3.1077258133776127, 3.004283768668074, 2.8985052335502246, 2.871118907821142, 2.929307536663373, 3.1493471673412046, 3.083387933004815, 2.981576156167705, 4.703124865771088, 2.942554955099183]

FID scores for DDPM samples compared to test set:
 [4.13994578184554, 4.070120928869358, 3.9259231495460085, 3.9175841449746827, 3.992885546565219, 4.15854527108263, 4.170864904351902, 4.0398808268002995, 5.7907602766106265, 4.011086961666933]

FID scores for DDPM reconstructions compared to train set:
 [3.037101460298402, 2.9942554774947894, 2.7851219783775036, 2.9253703397929485, 3.1037126021468, 3.289042837806562, 3.0659465327138378, 2.958097280214133, 5.013461221739021, 3.087500295145105]

FID scores for DDPM reconstructions compared to test set:
 [3.9866009325390905, 3.9307704417653895, 3.7077555674055134, 3.886290291013438, 4.043064551581153, 4.240176449089006, 3.995581644892468, 3.9076186059508586, 5.970705206578884, 4.040210361364586]

In [34]:
# compute average and standard deviation of FID scores for DDPM 
print("FashionMNIST")
print("-----------------------------------")
print("Reconstructions")
print("\nAverage FID score for DDPM reconstructions compared to test set:", np.mean(fmnist_test_FID_reconstructions_ddpm))
print("\nStandard deviation FID score for DDPM reconstructions compared to test set:", np.std(fmnist_test_FID_reconstructions_ddpm))
print("-----------------------------------")
print("Generations")
print("\nAverage FID score for DDPM samples compared to test set:", np.mean(fmnist_test_FID_generations_ddpm))
print("\nStandard deviation FID score for DDPM samples compared to test set:", np.std(fmnist_test_FID_generations_ddpm))

FashionMNIST
-----------------------------------
Reconstructions

Average FID score for DDPM reconstructions compared to test set: 4.170877405218039

Standard deviation FID score for DDPM reconstructions compared to test set: 0.6135720909093215
-----------------------------------
Generations

Average FID score for DDPM samples compared to test set: 4.22175977923132

Standard deviation FID score for DDPM samples compared to test set: 0.529898605954831


# CIFAR10

In [35]:
dataset = 'cifar10/'

# use config of some trained cifar10 model to load data correctly
ex_name = '20240303-144417_f2db4'

checkpoint_path = path+dataset+ex_name
with open(checkpoint_path + "/config.yaml", 'r') as stream:
    configs = yaml.load(stream,Loader=yaml.Loader)

# load data
trainset, trainset_eval, testset = get_data(configs)
gen_train = get_gen(trainset, configs, validation=False, shuffle=False)
gen_train_eval = get_gen(trainset_eval, configs, validation=True, shuffle=False)
gen_test = get_gen(testset, configs, validation=True, shuffle=False)
gen_train_eval_iter = iter(gen_train_eval)
gen_test_iter = iter(gen_test)
y_train = trainset_eval.dataset.targets.clone().detach()[trainset_eval.indices].numpy()
y_test = testset.dataset.targets.clone().detach()[testset.indices].numpy()

# setup device, mps is for M1 or M2 macs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "mps"

# precompute or load fid scores for train and test
data_stats_train = get_precomputed_fid_scores_path(trainset.dataset.data, configs['data']['data_name'],
                                                subset="train", device=device)
data_stats_test = get_precomputed_fid_scores_path(testset.dataset.data, configs['data']['data_name'], subset="test",
                                                device=device)

# paths to the generated images
base_path = '../from_cluster/results/'
seed_list = ["seed_1", "seed_2", "seed_3", "seed_4", "seed_5", "seed_6", "seed_7", "seed_8", "seed_9", "seed_10"]

# lists to store FID scores for each seed
cifar10_train_FID_generations_ddpm = []
cifar10_test_FID_generations_ddpm = []
cifar10_train_FID_reconstructions_ddpm = []
cifar10_test_FID_reconstructions_ddpm = []

for i, seed in enumerate(seed_list):
    ddpm_samples_path = base_path + dataset + seed + '/ddpm/sample/'
    ddpm_reconstructions_path = base_path + dataset + seed + '/ddpm/recons/'

    # load all images from the path and create a numpy array
    ddpm_samples = []
    for filename in os.listdir(ddpm_samples_path):
        if filename.endswith(".png"):
            img = Image.open(ddpm_samples_path + filename)
            img = np.array(img)
            ddpm_samples.append(img)
    ddpm_samples = np.array(ddpm_samples)

    ddpm_reconstructions = []
    for filename in os.listdir(ddpm_reconstructions_path):
        if filename.endswith(".png"):
            img = Image.open(ddpm_reconstructions_path + filename)
            img = np.array(img)
            ddpm_reconstructions.append(img)
    ddpm_reconstructions = np.array(ddpm_reconstructions)

    # precompute FID scores for generated and reconstructed images of DDPM

    # DDPM samples
    ddpm_stats_generations = save_fid_stats_as_dict(ddpm_samples, batch_size=256, device=device, dims=2048)
    train_FID_gen_ddpm = calculate_fid([data_stats_train, ddpm_stats_generations], batch_size=256, device=device, dims=2048)
    test_FID_gen_ddpm = calculate_fid([data_stats_test, ddpm_stats_generations], batch_size=256, device=device, dims=2048)
    cifar10_train_FID_generations_ddpm.append(train_FID_gen_ddpm)
    cifar10_test_FID_generations_ddpm.append(test_FID_gen_ddpm)

    # DDPM reconstructions
    ddpm_stats_reconstructions = save_fid_stats_as_dict(ddpm_reconstructions, batch_size=256, device=device, dims=2048)
    train_FID_recon_ddpm = calculate_fid([data_stats_train, ddpm_stats_reconstructions], batch_size=256, device=device, dims=2048)
    test_FID_recon_ddpm = calculate_fid([data_stats_test, ddpm_stats_reconstructions], batch_size=256, device=device, dims=2048)
    cifar10_train_FID_reconstructions_ddpm.append(train_FID_recon_ddpm)
    cifar10_test_FID_reconstructions_ddpm.append(test_FID_recon_ddpm)

Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified
Saving FID statistics


100%|██████████| 40/40 [00:52<00:00,  1.32s/it]


Saving FID statistics


100%|██████████| 40/40 [00:52<00:00,  1.30s/it]


Saving FID statistics


...

100%|██████████| 40/40 [00:50<00:00,  1.26s/it]


Saving FID statistics


100%|██████████| 40/40 [00:51<00:00,  1.29s/it]


Saving FID statistics


100%|██████████| 40/40 [00:51<00:00,  1.29s/it]


In [36]:
print("\nFID scores for DDPM samples compared to train set:\n", cifar10_train_FID_generations_ddpm)
print("\nFID scores for DDPM samples compared to test set:\n", cifar10_test_FID_generations_ddpm)
print("\nFID scores for DDPM reconstructions compared to train set:\n", cifar10_train_FID_reconstructions_ddpm)
print("\nFID scores for DDPM reconstructions compared to test set:\n", cifar10_test_FID_reconstructions_ddpm)


FID scores for DDPM samples compared to train set:
 [20.767625425869255, 19.682917802851193, 20.206558574944324, 20.658563434333246, 20.10795908708343, 20.531976722193747, 19.7069925060311, 20.308775853994575, 20.199873337598547, 20.013765784226734]

FID scores for DDPM samples compared to test set:
 [22.80768232044005, 21.746769476708437, 22.332262393935025, 22.75129312659027, 22.194330225228896, 22.58406256869779, 21.828081598545566, 22.370585011955995, 22.20464916626429, 22.05292356398894]

FID scores for DDPM reconstructions compared to train set:
 [13.595123356272552, 13.307016891534317, 13.85041181507836, 13.584125599412914, 13.213902942578784, 13.668355680474292, 13.660554740574469, 13.177101720224528, 14.123110253240895, 13.32288486875541]

FID scores for DDPM reconstructions compared to test set:
 [15.489439771148966, 15.179599848166902, 15.741093323416635, 15.429711186292252, 15.097947776905585, 15.554444696282076, 15.540960045883821, 15.081977160310203, 15.993967448506226, 

In [37]:
# compute average and standard deviation of FID scores for DDPM 
print("CIFAR-10")
print("-----------------------------------")
print("Reconstructions")
print("\nAverage FID score for DDPM reconstructions compared to test set:", np.mean(cifar10_test_FID_reconstructions_ddpm))
print("\nStandard deviation FID score for DDPM reconstructions compared to test set:", np.std(cifar10_test_FID_reconstructions_ddpm))
print("-----------------------------------")
print("Generations")
print("\nAverage FID score for DDPM samples compared to test set:", np.mean(cifar10_test_FID_generations_ddpm))
print("\nStandard deviation FID score for DDPM samples compared to test set:", np.std(cifar10_test_FID_generations_ddpm))

CIFAR-10
-----------------------------------
Reconstructions

Average FID score for DDPM reconstructions compared to test set: 15.42944866498065

Standard deviation FID score for DDPM reconstructions compared to test set: 0.28286070278369657
-----------------------------------
Generations

Average FID score for DDPM samples compared to test set: 22.287263945235527

Standard deviation FID score for DDPM samples compared to test set: 0.3400845542147974
